In [1]:
import sys
from pathlib import Path
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments
from datasets import load_dataset, load_from_disk

# Setup Path per importare il modulo src
ROOT = Path.cwd().resolve().parent
if str(ROOT / "src") not in sys.path:
    sys.path.append(str(ROOT / "src"))

from project_paths import get_paths
from distillation import TinyBertForDistillation, DistillationTrainer

paths = get_paths(ROOT)

c:\Users\cola0\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
TEACHER_CHECKPOINT = paths.checkpoints / "teacher_bert_large_llrd" / "checkpoint-6000" 
STUDENT_CHECKPOINT = "google/bert_uncased_L-4_H-512_A-8"

TEMP = 5.0          # Temperatura (Hinton)
ALPHA = 0.5         # 50% Soft Loss, 50% Hard Loss
LOSS_WEIGHTS = {
    "pred": 1.0,    # Peso Prediction
    "emb":  1.0,    # Peso Embedding
    "hidden": 1.0,  # Peso Hidden States
    "attn": 1.0     # Peso Attention
}

In [3]:
# --- 1. CARICAMENTO DATI ---
print("📂 Caricamento Dataset...")
# Assumiamo che tu abbia i file .parquet processati (con Head+Tail)
train_ds = load_dataset("parquet", data_files=str(paths.data_processed / "train_120k_ht.parquet"))["train"]
val_ds = load_dataset("parquet", data_files=str(paths.data_processed / "val_120k_ht.parquet"))["train"]

# --- 2. PREPARAZIONE MODELLI ---
print(f"🏗️ Caricamento Teacher: {TEACHER_CHECKPOINT}")
teacher_model = AutoModelForSequenceClassification.from_pretrained(
    TEACHER_CHECKPOINT, num_labels=1
).to("cuda")

print(f"🏗️ Inizializzazione Student: {STUDENT_CHECKPOINT}")
# Carichiamo il modello base HuggingFace
student_base = AutoModelForSequenceClassification.from_pretrained(
    STUDENT_CHECKPOINT, num_labels=1
)
# Avvolgiamo lo student nel nostro wrapper che aggiunge le proiezioni
student_wrapper = TinyBertForDistillation(
    student_model=student_base,
    teacher_hidden_size=1024 # Dimensione del Teacher Large
).to("cuda")

# Tokenizer (usiamo quello del Teacher/Student che è uguale: BERT Base Uncased)
tokenizer = AutoTokenizer.from_pretrained("bert-large-uncased")

📂 Caricamento Dataset...
🏗️ Caricamento Teacher: C:\Users\cola0\Desktop\nlp.project-Colangelo-2526\checkpoints\teacher_bert_large_llrd\checkpoint-6000
🏗️ Inizializzazione Student: google/bert_uncased_L-4_H-512_A-8


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at google/bert_uncased_L-4_H-512_A-8 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [4]:
# --- FIX DATASET ---
# Rimuoviamo le colonne che non sono tensori (come 'movie_id', 'text', ecc.)
# Il modello vuole SOLO: input_ids, attention_mask, labels (e token_type_ids se c'è)

cols_to_keep = ["input_ids", "attention_mask", "labels", "token_type_ids"]

# Funzione per pulire il dataset mantenendo solo le colonne necessarie
def clean_dataset(ds):
    # Troviamo quali colonne extra ci sono
    extra_cols = [c for c in ds.column_names if c not in cols_to_keep]
    if extra_cols:
        print(f"🧹 Rimozione colonne non necessarie: {extra_cols}")
        return ds.remove_columns(extra_cols)
    return ds

# Applichiamo la pulizia
train_ds = clean_dataset(train_ds)
val_ds = clean_dataset(val_ds)

# Controllo finale
print(f"✅ Colonne Train finali: {train_ds.column_names}")

🧹 Rimozione colonne non necessarie: ['review_id', 'movie_id', 'text', 'label']
🧹 Rimozione colonne non necessarie: ['review_id', 'movie_id', 'text', 'label']
✅ Colonne Train finali: ['input_ids', 'attention_mask', 'token_type_ids']


In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score
from transformers import DataCollatorWithPadding
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# --- 1. FUNZIONE METRICHE (Per monitorare lo Student) ---
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    # predictions sono i logits
    # labels sono le ground truth (0 o 1)
    
    # Applichiamo Sigmoide (perché num_labels=1)
    probs = 1 / (1 + np.exp(-predictions))
    
    # Soglia 0.5
    preds = (probs > 0.5).astype(int).reshape(-1)
    labels = labels.reshape(-1)
    
    # Calcolo Metriche
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary')
    acc = accuracy_score(labels, preds)
    try:
        auc = roc_auc_score(labels, probs)
    except:
        auc = 0.5
        
    return {
        'accuracy': acc,
        'f1': f1,
        'precision': precision,
        'recall': recall,
        'auc': auc
    }

# --- 2. CONFIGURAZIONE TRAINING OTTIMIZZATA ---

# Se hai una 3070 (8GB VRAM), con TinyBERT puoi osare molto sul batch size.
# BERT-Large in eval mode consuma memoria, ma proviamo a spingere.
BATCH_SIZE = 32  # Proviamo 32. Se regge, prova 64!
GRADIENT_ACCUMULATION = 1 

training_args = TrainingArguments(
    output_dir=str(paths.checkpoints / "tinybert_ablation3_full"),
    
    # --- VELOCITÀ ---
    num_train_epochs=1,              # 3 epoche bastano (circa 9000 step totali con batch 32)
    per_device_train_batch_size=BATCH_SIZE, 
    per_device_eval_batch_size=BATCH_SIZE * 2, # In eval possiamo raddoppiare (niente gradienti)
    dataloader_num_workers=4,        # Velocizza il caricamento dati CPU -> GPU
    fp16=True,                       # Fondamentale per RTX 3070
    
    # --- LOGGING & EVAL ---
    logging_steps=100,
    eval_strategy="steps",
    eval_steps=50,                 # Valuta meno spesso per non perdere tempo
    save_steps=50,
    save_total_limit=2,              # Tieni solo gli ultimi 2 checkpoint per spazio
    
    # --- ALTRO ---
    learning_rate=5e-5,
    warmup_steps=50,
    weight_decay=0.01,
    report_to="none",
    remove_unused_columns=False
)

# --- 3. LANCIO DEL TRAINER ---
trainer = DistillationTrainer(
    teacher_model=teacher_model,
    model=student_wrapper,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics, # <--- Aggiunta qui
    alpha=ALPHA,
    temperature=TEMP,
    weights=LOSS_WEIGHTS
)

print(f"🚀 Avvio Distillation Veloce (Batch: {BATCH_SIZE})...")
trainer.train()

🚀 Avvio Distillation Veloce (Batch: 32)...


C:\Users\cola0\Desktop\nlp.project-Colangelo-2526\src\distillation.py:39: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `DistillationTrainer.__init__`. Use `processing_class` instead.
  super().__init__(*args, **kwargs)
BertSdpaSelfAttention is used but `torch.nn.functional.scaled_dot_product_attention` does not support non-absolute `position_embedding_type` or `output_attentions=True` or `head_mask`. Falling back to the manual attention implementation, but specifying the manual implementation will be required from Transformers version v5.0.0 onwards. This warning can be removed using the argument `attn_implementation="eager"` when loading the model.


Step,Training Loss,Validation Loss
